# Proyecto 3 · Generador de noticias en tiempo real en Python

**IA en la Era de la Computadora en la Nube** · Mgs. Rafael Acosta Torres

Construiremos un sistema que **vigila fuentes de noticias**, detecta titulares nuevos y **redacta un boletín** con IA, citando siempre las fuentes:

| Parte | Qué hacemos | Herramienta |
|---|---|---|
| A | Leer fuentes RSS en vivo | `requests` + `feedparser` |
| B | Limpiar, eliminar duplicados y ordenar | `difflib`, `pandas` |
| C | Redactar el boletín con citas | Gemini |
| D | Tiempo real: sondear cada *N* minutos y publicar en HTML | Bucle con estado |
| E | Alternativa: Gemini con búsqueda de Google | *Grounding* |

> ⚖️ **Ética periodística:** el sistema **resume y enlaza**; no copia artículos completos, no inventa hechos y marca el texto como generado con IA. Antes de publicar, una persona revisa.

### Dos modos de ejecución

* **Modo real (API de Google Gemini):** se activa solo cuando hay una clave. La clave se crea gratis en **Google AI Studio** (`aistudio.google.com` → *Get API key*) y **nunca se escribe en el cuaderno**:
  1. **Secretos de Colab** (recomendado): ícono de llave 🔑 → *Agregar secreto* → nombre `GEMINI_API_KEY` → pegar la clave → activar *Acceso al cuaderno*.
  2. **Campo oculto:** si no hay secreto, la celda de configuración la pide sin mostrarla. Enter vacío = simulación.
* **Modo simulación:** sin clave, sin costo y sin depender del servicio. El "modelo" se reemplaza por reglas locales para mostrar el **flujo** de la aplicación. Úselo como plan B en clase (`FORZAR_SIMULACION = True`).

> 🔒 No pegue la clave en una celda de código ni la muestre en pantalla: cualquiera que la vea puede usarla.

In [1]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
import os, sys
from getpass import getpass

FORZAR_SIMULACION = False          # True: usa el simulador aunque haya clave (plan B sin internet)
MODELO = "gemini-3.8-flash"        # alternativa más económica: "gemini-3.5-flash-lite"
EN_COLAB = "google.colab" in sys.modules

API_KEY = None
if not FORZAR_SIMULACION:
    if EN_COLAB:
        from google.colab import userdata
        for nombre in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
            try:
                API_KEY = userdata.get(nombre)
            except Exception:
                API_KEY = None
            if API_KEY:
                break
    else:
        API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not API_KEY and EN_COLAB:
        API_KEY = getpass("Pegue su clave de Gemini (Enter vacío = simulación): ").strip() or None

MODO_SIMULACION = API_KEY is None
print("Modo:", "SIMULACIÓN (sin API)" if MODO_SIMULACION else f"REAL · modelo {MODELO}")

Modo: SIMULACIÓN (sin API)


In [2]:
if EN_COLAB:
    !pip -q install -U feedparser google-genai

In [3]:
# Conexión con Gemini (solo en modo real)
if not MODO_SIMULACION:
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=API_KEY)
    del API_KEY                                   # no dejar la clave en una variable visible
    try:
        client.models.get(model=MODELO)           # prueba mínima (no consume tokens)
        print(f"✅ Conexión con Gemini verificada · modelo {MODELO}")
    except Exception as e:
        print(f"⚠️  No se pudo usar la clave o el modelo ({type(e).__name__}): {str(e)[:200]}")
        print("   Se continúa en MODO SIMULACIÓN.")
        MODO_SIMULACION = True

---
## Parte A · Leer fuentes RSS en vivo

**RSS** es un formato XML que publican los medios con sus titulares más recientes (título, enlace, fecha, resumen). Es la forma más simple y respetuosa de obtener noticias en tiempo real: lo ofrece el propio medio.

Usamos la búsqueda de **Google Noticias** en formato RSS, que agrega muchos medios, y puede añadir los RSS de los medios que prefiera. Si no hay internet, el cuaderno usa un **RSS de ejemplo ficticio** para que la clase continúe.

In [4]:
import re, time, html, difflib
from datetime import datetime, timedelta, timezone
from urllib.parse import quote_plus
import requests, feedparser
import pandas as pd

TEMA = "inteligencia artificial Colombia"
FUENTES = {
    "Google Noticias": f"https://news.google.com/rss/search?q={quote_plus(TEMA)}&hl=es-419&gl=CO&ceid=CO:es-419",
    # "Mi medio favorito": "https://ejemplo.com/rss",
}

def rss_de_ejemplo(n=10, desplazamiento_min=0):
    # RSS ficticio para trabajar sin internet (titulares inventados, claramente marcados)
    ahora = datetime.now(timezone.utc) - timedelta(minutes=desplazamiento_min)
    titulares = [
        "Universidad de la región abre laboratorio de IA en la nube para estudiantes",
        "Empresas del Caribe aceleran la adopción de asistentes conversacionales",
        "Nuevo programa de becas para formación en ciencia de datos",
        "Gobierno local presenta guía para el uso responsable de la IA",
        "Startups colombianas usan visión artificial para controlar la calidad de alimentos",
        "Universidad de la región abre un laboratorio de IA en la nube para estudiantes",
        "Hospitales prueban OCR para digitalizar historias clínicas en papel",
        "Estudio estima el consumo energético de los centros de datos en el país",
        "Puerto de la ciudad usa modelos predictivos para reducir tiempos de espera",
        "Expertos piden reglas claras sobre derechos de autor y contenido generado por IA",
    ][:n]
    items = "".join(
        f"<item><title>{t} - Diario de Ejemplo (ficticio)</title>"
        f"<link>https://ejemplo.org/noticia/{k}</link>"
        f"<pubDate>{(ahora - timedelta(minutes=17 * (len(titulares) - 1 - k))).strftime('%a, %d %b %Y %H:%M:%S +0000')}</pubDate>"
        f"<source>Diario de Ejemplo (ficticio)</source></item>"
        for k, t in enumerate(titulares))
    return f"<?xml version='1.0'?><rss><channel><title>Ejemplo</title>{items}</channel></rss>"

def leer_fuente(nombre, url, timeout=10):
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0 (clase de IA)"})
        r.raise_for_status()
        feed = feedparser.parse(r.content)
        if not feed.entries:
            raise ValueError("RSS vacío")
        origen = "en vivo"
    except Exception as e:
        print(f"⚠️  {nombre}: sin conexión ({type(e).__name__}); se usa el RSS de ejemplo ficticio.")
        feed, origen = feedparser.parse(rss_de_ejemplo()), "ejemplo"
    noticias = []
    for e in feed.entries:
        fecha = datetime(*e.published_parsed[:6], tzinfo=timezone.utc) if e.get("published_parsed") else None
        fuente = e.get("source", {}).get("title") if isinstance(e.get("source"), dict) else None
        noticias.append({"titulo": html.unescape(e.get("title", "")), "enlace": e.get("link", ""),
                         "fecha": fecha, "fuente": fuente or nombre, "origen": origen})
    return noticias

crudas = [n for nombre, url in FUENTES.items() for n in leer_fuente(nombre, url)]
df = pd.DataFrame(crudas)
print(f"{len(df)} titulares leídos · origen: {df.origen.unique().tolist()}")
df[["fecha", "fuente", "titulo"]].head(8)

⚠️  Google Noticias: sin conexión (ProxyError); se usa el RSS de ejemplo ficticio.
10 titulares leídos · origen: ['ejemplo']


,fecha,fuente,titulo
0,2026-10-29 15:28:14+00:00,Diario de Ejemplo (ficticio),Universidad de la región abre laboratorio de I...
1,2026-10-29 15:45:14+00:00,Diario de Ejemplo (ficticio),Empresas del Caribe aceleran la adopción de as...
2,2026-10-29 16:02:14+00:00,Diario de Ejemplo (ficticio),Nuevo programa de becas para formación en cien...
3,2026-10-29 16:19:14+00:00,Diario de Ejemplo (ficticio),Gobierno local presenta guía para el uso respo...
4,2026-10-29 16:36:14+00:00,Diario de Ejemplo (ficticio),Startups colombianas usan visión artificial pa...
5,2026-10-29 16:53:14+00:00,Diario de Ejemplo (ficticio),Universidad de la región abre un laboratorio d...
6,2026-10-29 17:10:14+00:00,Diario de Ejemplo (ficticio),Hospitales prueban OCR para digitalizar histor...
7,2026-10-29 17:27:14+00:00,Diario de Ejemplo (ficticio),Estudio estima el consumo energético de los ce...


---
## Parte B · Limpiar, eliminar duplicados y ordenar

* Google Noticias agrega el nombre del medio al final del título (`... - El Medio`): lo separamos.
* El mismo hecho aparece en varios medios con títulos casi iguales: los agrupamos con la **similitud de secuencias** de `difflib` (1 = idénticos).
* Ordenamos del más reciente al más antiguo y nos quedamos con los *N* primeros.

In [5]:
def limpiar(noticias):
    for n in noticias:
        m = re.match(r"^(.*)\s+-\s+([^-]+)$", n["titulo"])
        if m:
            n["titulo"], n["fuente"] = m.group(1).strip(), m.group(2).strip()
    return noticias

def similitud(a, b):
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()

def sin_duplicados(noticias, umbral=0.85):
    unicas = []
    for n in sorted(noticias, key=lambda x: x["fecha"] or datetime.min.replace(tzinfo=timezone.utc), reverse=True):
        if all(similitud(n["titulo"], u["titulo"]) < umbral for u in unicas):
            unicas.append(n)
    return unicas

N_MAX = 8
noticias = sin_duplicados(limpiar(crudas))[:N_MAX]
print(f"Después de limpiar y quitar duplicados: {len(noticias)} noticias")
for k, n in enumerate(noticias, 1):
    hora = n["fecha"].astimezone(timezone(timedelta(hours=-5))).strftime("%d/%m %H:%M") if n["fecha"] else "s. f."
    print(f"[{k}] {hora} · {n['fuente']}: {n['titulo']}")

Después de limpiar y quitar duplicados: 8 noticias
[1] 29/10 13:01 · Diario de Ejemplo (ficticio): Expertos piden reglas claras sobre derechos de autor y contenido generado por IA
[2] 29/10 12:44 · Diario de Ejemplo (ficticio): Puerto de la ciudad usa modelos predictivos para reducir tiempos de espera
[3] 29/10 12:27 · Diario de Ejemplo (ficticio): Estudio estima el consumo energético de los centros de datos en el país
[4] 29/10 12:10 · Diario de Ejemplo (ficticio): Hospitales prueban OCR para digitalizar historias clínicas en papel
[5] 29/10 11:53 · Diario de Ejemplo (ficticio): Universidad de la región abre un laboratorio de IA en la nube para estudiantes
[6] 29/10 11:36 · Diario de Ejemplo (ficticio): Startups colombianas usan visión artificial para controlar la calidad de alimentos
[7] 29/10 11:19 · Diario de Ejemplo (ficticio): Gobierno local presenta guía para el uso responsable de la IA
[8] 29/10 11:02 · Diario de Ejemplo (ficticio): Nuevo programa de becas para formación en cie

---
## Parte C · Redactar el boletín con IA (y citar las fuentes)

El modelo solo recibe **los titulares que leímos** y debe citar cada afirmación con su número `[n]`. La lista de enlaces la arma **el código**, no el modelo: así los enlaces nunca se inventan.

In [6]:
from IPython.display import Markdown, display

COLOMBIA = timezone(timedelta(hours=-5))

def construir_prompt(noticias, tema):
    lista = "\n".join(f"[{k}] {n['titulo']} ({n['fuente']})" for k, n in enumerate(noticias, 1))
    return (f"Eres editor de un boletín de noticias sobre «{tema}». Con SOLO estos titulares, "
            f"escribe un boletín breve en español: un titular general, un párrafo de contexto y "
            f"3 a 5 viñetas. Cada afirmación debe terminar con la cita [n] del titular que la respalda. "
            f"No agregues datos que no estén en los titulares. No incluyas enlaces.\n\nTitulares:\n{lista}")

def redactar_simulado(noticias, tema):
    vinetas = "\n".join(f"- {n['titulo']} [{k}]." for k, n in enumerate(noticias[:5], 1))
    return (f"**Lo último sobre {tema}**\n\n"
            f"Estas son las {min(5, len(noticias))} noticias más recientes que encontramos:\n\n{vinetas}")

def redactar_boletin(noticias, tema):
    if MODO_SIMULACION:
        return "*[simulación: texto armado con plantillas]*\n\n" + redactar_simulado(noticias, tema)
    try:
        r = client.models.generate_content(model=MODELO, contents=construir_prompt(noticias, tema))
        return r.text or ""
    except Exception as e:
        return f"(Error del servicio: {type(e).__name__}) " + redactar_simulado(noticias, tema)

def citas_validas(texto, n):
    usadas = {int(x) for x in re.findall(r"\[(\d+)\]", texto)}
    return usadas, {u for u in usadas if not 1 <= u <= n}

def boletin_completo(noticias, tema):
    cuerpo = redactar_boletin(noticias, tema)
    usadas, fuera = citas_validas(cuerpo, len(noticias))
    fuentes = "\n".join(f"{k}. [{n['titulo']}]({n['enlace']}) — {n['fuente']}" for k, n in enumerate(noticias, 1))
    corte = datetime.now(COLOMBIA).strftime("%d/%m/%Y %H:%M")
    aviso = f"\n\n⚠️ Citas fuera de rango: {sorted(fuera)}" if fuera else ""
    return (f"{cuerpo}{aviso}\n\n---\n**Fuentes** (corte: {corte}, hora de Colombia)\n\n{fuentes}\n\n"
            f"*Boletín generado automáticamente con IA a partir de titulares públicos; verifique en la fuente original.*")

boletin = boletin_completo(noticias, TEMA)
display(Markdown(boletin))

*[simulación: texto armado con plantillas]*

**Lo último sobre inteligencia artificial Colombia**

Estas son las 5 noticias más recientes que encontramos:

- Expertos piden reglas claras sobre derechos de autor y contenido generado por IA [1].
- Puerto de la ciudad usa modelos predictivos para reducir tiempos de espera [2].
- Estudio estima el consumo energético de los centros de datos en el país [3].
- Hospitales prueban OCR para digitalizar historias clínicas en papel [4].
- Universidad de la región abre un laboratorio de IA en la nube para estudiantes [5].

---
**Fuentes** (corte: 29/10/2026 13:01, hora de Colombia)

1. [Expertos piden reglas claras sobre derechos de autor y contenido generado por IA](https://ejemplo.org/noticia/9) — Diario de Ejemplo (ficticio)
2. [Puerto de la ciudad usa modelos predictivos para reducir tiempos de espera](https://ejemplo.org/noticia/8) — Diario de Ejemplo (ficticio)
3. [Estudio estima el consumo energético de los centros de datos en el país](https://ejemplo.org/noticia/7) — Diario de Ejemplo (ficticio)
4. [Hospitales prueban OCR para digitalizar historias clínicas en papel](https://ejemplo.org/noticia/6) — Diario de Ejemplo (ficticio)
5. [Universidad de la región abre un laboratorio de IA en la nube para estudiantes](https://ejemplo.org/noticia/5) — Diario de Ejemplo (ficticio)
6. [Startups colombianas usan visión artificial para controlar la calidad de alimentos](https://ejemplo.org/noticia/4) — Diario de Ejemplo (ficticio)
7. [Gobierno local presenta guía para el uso responsable de la IA](https://ejemplo.org/noticia/3) — Diario de Ejemplo (ficticio)
8. [Nuevo programa de becas para formación en ciencia de datos](https://ejemplo.org/noticia/2) — Diario de Ejemplo (ficticio)

*Boletín generado automáticamente con IA a partir de titulares públicos; verifique en la fuente original.*

---
## Parte D · Tiempo real: sondear, detectar lo nuevo y publicar

"Tiempo real" en este contexto significa **sondear** (*polling*) las fuentes cada cierto tiempo:

1. Leer los RSS.
2. Comparar con los enlaces **ya vistos** (estado del sistema).
3. Si hay noticias nuevas, **regenerar el boletín** y publicarlo (aquí, un archivo `boletin.html`).
4. Esperar *N* minutos y repetir.

> No sondee más seguido de lo necesario (cada 10–15 minutos es razonable): cada lectura consume recursos del medio, y cada boletín consume tokens.

In [7]:
try:
    import markdown as md_lib            # convierte el boletín a HTML (incluido en Colab)
except ImportError:
    !pip -q install markdown
    import markdown as md_lib

def feedparser_a_lista(xml):
    return [{"titulo": e.title, "enlace": e.link, "fuente": "Diario de Ejemplo (ficticio)", "origen": "ejemplo",
             "fecha": datetime(*e.published_parsed[:6], tzinfo=timezone.utc)} for e in feedparser.parse(xml).entries]

class MonitorNoticias:
    def __init__(self, fuentes, tema, n_max=8):
        self.fuentes, self.tema, self.n_max = fuentes, tema, n_max
        self.vistos = set()
        self.historial = []

    def ronda(self, lector=None):
        lector = lector or (lambda: [n for nom, url in self.fuentes.items() for n in leer_fuente(nom, url)])
        actuales = sin_duplicados(limpiar(lector()))[: self.n_max]
        nuevas = [n for n in actuales if n["enlace"] not in self.vistos]
        self.vistos |= {n["enlace"] for n in actuales}
        hora = datetime.now(COLOMBIA).strftime("%H:%M:%S")
        if not nuevas:
            print(f"{hora} · sin novedades")
            return None
        print(f"{hora} · {len(nuevas)} noticias nuevas → se regenera el boletín")
        for n in nuevas[:3]:
            print(f"      + {n['titulo'][:90]}")
        texto = boletin_completo(actuales, self.tema)
        with open("boletin.html", "w", encoding="utf-8") as f:
            f.write(f"<meta charset='utf-8'><title>Boletín · {self.tema}</title>"
                    f"<body style='font-family:sans-serif;max-width:720px;margin:auto'>"
                    f"{md_lib.markdown(texto)}</body>")
        self.historial.append((hora, len(nuevas)))
        return texto

monitor = MonitorNoticias(FUENTES, TEMA)

RONDAS, INTERVALO_SEG = 3, 600          # en clase: 3 rondas cada 10 minutos
if MODO_SIMULACION or df.origen.eq("ejemplo").all():
    # Demostración acelerada sin internet: el RSS de ejemplo "publica" noticias nuevas en cada ronda
    INTERVALO_SEG = 2
    lectores = [lambda k=k: limpiar(feedparser_a_lista(rss_de_ejemplo(n=4 + 3 * k))) for k in range(RONDAS)]
else:
    lectores = [None] * RONDAS

for k in range(RONDAS):
    monitor.ronda(lectores[k])
    if k < RONDAS - 1:
        time.sleep(INTERVALO_SEG)
print("\nBoletín publicado en boletin.html · actualizaciones:", monitor.historial)

13:01:14 · 4 noticias nuevas → se regenera el boletín
      + Gobierno local presenta guía para el uso responsable de la IA
      + Nuevo programa de becas para formación en ciencia de datos
      + Empresas del Caribe aceleran la adopción de asistentes conversacionales


13:01:16 · 3 noticias nuevas → se regenera el boletín
      + Hospitales prueban OCR para digitalizar historias clínicas en papel
      + Universidad de la región abre un laboratorio de IA en la nube para estudiantes
      + Startups colombianas usan visión artificial para controlar la calidad de alimentos


13:01:18 · 3 noticias nuevas → se regenera el boletín
      + Expertos piden reglas claras sobre derechos de autor y contenido generado por IA
      + Puerto de la ciudad usa modelos predictivos para reducir tiempos de espera
      + Estudio estima el consumo energético de los centros de datos en el país

Boletín publicado en boletin.html · actualizaciones: [('13:01:14', 4), ('13:01:16', 3), ('13:01:18', 3)]


---
## Parte E · Alternativa: Gemini con búsqueda de Google (*grounding*)

En lugar de leer RSS, el modelo puede **buscar en Google** por su cuenta y devolver las fuentes que usó. Es más simple de programar, pero se controla menos qué medios se consultan, y cada búsqueda tiene un costo (el nivel gratuito tiene un límite diario).

In [8]:
if MODO_SIMULACION:
    print("Parte E requiere una clave de Gemini (vea la celda de configuración).")
else:
    try:
        r = client.models.generate_content(
            model=MODELO,
            contents=f"Resume en 5 viñetas las noticias más recientes (últimas 24 horas) sobre {TEMA}. "
                     f"Indica la fecha de cada una.",
            config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())]))
        display(Markdown(r.text or ""))
        meta = r.candidates[0].grounding_metadata
        if meta and meta.grounding_chunks:
            print("Fuentes consultadas:")
            for k, ch in enumerate(meta.grounding_chunks, 1):
                if ch.web:
                    print(f"  {k}. {ch.web.title} — {ch.web.uri}")
        if meta and meta.web_search_queries:
            print("Búsquedas realizadas:", meta.web_search_queries)
    except Exception as e:
        print(f"No disponible ({type(e).__name__}): {str(e)[:200]}")

Parte E requiere una clave de Gemini (vea la celda de configuración).


---
## Del cuaderno a la nube (conexión con la Unidad 1)

Un cuaderno de Colab se detiene cuando se cierra. Para que el monitor funcione **24/7**:

| Opción | Cómo se programa | Modelo de servicio |
|---|---|---|
| Cloud Run Jobs / Cloud Functions + **Cloud Scheduler** | Una ejecución de `ronda()` cada 15 min | Serverless (PaaS/FaaS) |
| AWS Lambda + **EventBridge** | Regla `rate(15 minutes)` | Serverless |
| GitHub Actions | `schedule: cron: "*/15 * * * *"` | CI/CD gratuito (límites) |
| Máquina virtual con `cron` | `*/15 * * * * python monitor.py` | IaaS |

Guarde `vistos` en un almacenamiento persistente (un objeto en S3/GCS o una base de datos): cada ejecución serverless empieza "en blanco".

## Ejercicios

1. **Su tema:** cambie `TEMA` por un tema de su carrera y agregue el RSS de un medio regional.
2. **Alertas:** si un titular nuevo contiene una palabra clave (p. ej. "convocatoria"), imprima una alerta destacada.
3. **Verificación de citas:** pida al modelo una cita inexistente (por ejemplo, modifique el *prompt*) y compruebe que `citas_validas` la detecta.
4. **Costo:** estime cuántos tokens consume un boletín y cuánto costaría generar uno cada 15 minutos durante un mes.

*Los titulares del RSS de ejemplo son ficticios.*